In [ ]:
# Public API surface: top-level classes and version are present.
from py4j.java_gateway import JavaGateway, GatewayParameters, CallbackServerParameters, GatewayClient
from py4j.protocol import Py4JError, Py4JNetworkError, Py4JJavaError

for cls in (JavaGateway, GatewayParameters, CallbackServerParameters, GatewayClient):
    assert isinstance(cls, type), f"{cls!r} should be a class"

for exc in (Py4JError, Py4JNetworkError, Py4JJavaError):
    assert issubclass(exc, Exception), f"{exc!r} should subclass Exception"

print("JavaGateway:", JavaGateway)
print("GatewayParameters:", GatewayParameters)
print("Py4JError MRO depth:", len(Py4JError.__mro__))


In [ ]:
# GatewayParameters: defaults and customization (no JVM required).
from py4j.java_gateway import GatewayParameters, DEFAULT_PORT, DEFAULT_ADDRESS

p_default = GatewayParameters()
assert p_default.port == DEFAULT_PORT
assert p_default.address == DEFAULT_ADDRESS
assert p_default.auto_convert is False
assert p_default.eager_load is False

p_custom = GatewayParameters(address='192.0.2.1', port=12345, auto_convert=True, auto_field=True)
assert p_custom.address == '192.0.2.1'
assert p_custom.port == 12345
assert p_custom.auto_convert is True
assert p_custom.auto_field is True

print(f"DEFAULT_PORT={DEFAULT_PORT}, DEFAULT_ADDRESS={DEFAULT_ADDRESS}")
print(f"custom: {p_custom.address}:{p_custom.port}")


In [ ]:
# Protocol layer: command/escape helpers operate on pure strings.
from py4j import protocol

# Escape and unescape are inverses for typical inputs.
samples = ["hello", "line1\nline2", "with\ttab", "with\\backslash", ""]
for s in samples:
    escaped = protocol.escape_new_line(s)
    restored = protocol.unescape_new_line(escaped)
    assert restored == s, f"round-trip failed for {s!r}: got {restored!r}"
    assert '\n' not in escaped, f"escaped form still contains newline: {escaped!r}"

# Command-part encoding produces the documented type-prefixed, newline-terminated form.
encoded = protocol.get_command_part(42)
assert isinstance(encoded, str)
assert encoded.startswith(protocol.INTEGER_TYPE)
assert encoded.endswith('\n')
print("escape round-trip OK; get_command_part(42) =", repr(encoded))

In [ ]:
# GatewayClient connect failure surfaces as Py4JNetworkError (no JVM listening).
import socket

from py4j.java_gateway import GatewayClient, GatewayParameters
from py4j.protocol import Py4JNetworkError

# Pick a port that is almost certainly closed by binding+closing.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(('127.0.0.1', 0))
    closed_port = s.getsockname()[1]

client = GatewayClient(gateway_parameters=GatewayParameters(address='127.0.0.1', port=closed_port))
assert client.gateway_parameters.port == closed_port

raised = False
try:
    client.send_command("x")
except Py4JNetworkError:
    # Only the network error counts: py4j wraps a refused connection in Py4JNetworkError,
    # so any other exception means the client is broken and fails this cell.
    raised = True

assert raised, "expected a network/connection error when no JVM is listening"
print("connection-refused path raised as expected")


In [ ]:
# version + module geography sanity (no JVM required).
import py4j
import py4j.version
from py4j import compat, finalizer, java_collections, java_gateway, protocol, signals

# version submodule must expose a version string consistent with package attribute (if any).
ver = py4j.version.__version__
assert isinstance(ver, str) and len(ver) > 0
print(f"py4j.version.__version__ = {ver}")

# Each submodule we touched is a real module object.
import types
for mod in (compat, finalizer, java_collections, java_gateway, protocol, signals):
    assert isinstance(mod, types.ModuleType), f"{mod!r} is not a module"

# finalizer exposes the Finalizer / ThreadSafeFinalizer machinery.
assert hasattr(finalizer, 'Finalizer')
assert hasattr(finalizer, 'ThreadSafeFinalizer')

# signals has the Signal class used by callback hooks.
assert hasattr(signals, 'Signal')

# The Java half ships as a data file (share/py4j/py4j<version>.jar). find_jar_path() returns
# an empty string when no jar can be found.
import os
jar_path = java_gateway.find_jar_path()
assert os.path.isfile(jar_path), f"py4j jar not found: find_jar_path() returned {jar_path!r}"
print(f"py4j jar: {os.path.basename(jar_path)}")

print("py4j top-level surface (version, finalizer, signals, java_collections) OK")
